[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/gw-lvk-python-course/blob/main/Course/Problema_Ondas_Gravitacionales_Bayesiano.ipynb)


# Problema aplicado — Inferencia bayesiana de masa chirp y distancia
**Gravitational-Wave Data Analysis with Python** · LVK Python Course

En este notebook resolvemos un problema realista: inferir la **masa chirp** $\mathcal{M}$ y la **distancia luminosidad** $d_L$ de un evento tipo GW150914 usando estadística bayesiana y un modelo físico simple del inspiral.


---
## 0. Instalación de dependencias
Usaremos un stack mínimo para el análisis numérico y la visualización.


In [ ]:
import subprocess, sys
packages = ["numpy", "scipy", "matplotlib"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)


---
## 1. Preparación del entorno y constantes físicas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import G, c, pi

M_SUN = 1.98847e30  # kg
MPC = 3.085677581e22  # m

np.set_printoptions(precision=4, suppress=True)


---
## 2. Modelo físico del inspiral (Newtoniano)


In [ ]:
def t_chirp(f_hz, mc_solar):
    """Tiempo antes de la coalescencia para una frecuencia f (SI)."""
    mc = mc_solar * M_SUN
    return 5 / 256 * (G * mc / c**3) ** (-5/3) * (pi * f_hz) ** (-8/3)

def h_amp(f_hz, mc_solar, d_mpc):
    """Amplitud aproximada de la señal inspiral (orientación óptima)."""
    mc = mc_solar * M_SUN
    d = d_mpc * MPC
    return 4 * (G * mc) ** (5/3) * (pi * f_hz) ** (2/3) / (c**4 * d)


---
## 3. Simulación de datos (frecuencia, tiempo y amplitud observada)


In [ ]:
rng = np.random.default_rng(42)

freq = np.linspace(30, 150, 15)

mc_true = 28.0  # M_sun
d_true = 420.0  # Mpc

t_true = t_chirp(freq, mc_true)
h_true = h_amp(freq, mc_true, d_true)

sigma_t = 0.02  # s
sigma_h = 0.15 * h_true

t_obs = t_true + rng.normal(0, sigma_t, size=freq.size)
h_obs = h_true + rng.normal(0, sigma_h)

t_obs[:5], h_obs[:5]


---
## 4. Visualización de las mediciones


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(freq, t_true, label="Modelo (verdadero)")
axes[0].errorbar(freq, t_obs, yerr=sigma_t, fmt="o", label="Observado")
axes[0].set_xlabel("Frecuencia [Hz]")
axes[0].set_ylabel("t(f) [s antes de la coalescencia]")
axes[0].set_title("Relación tiempo–frecuencia")
axes[0].legend()

axes[1].plot(freq, h_true, label="Modelo (verdadero)")
axes[1].errorbar(freq, h_obs, yerr=sigma_h, fmt="o", label="Observado")
axes[1].set_xlabel("Frecuencia [Hz]")
axes[1].set_ylabel("Amplitud h(f)")
axes[1].set_title("Amplitud observada")
axes[1].legend()

plt.tight_layout()


---
## 5. Inferencia bayesiana en una malla $(\mathcal{M}, d_L)$


In [ ]:
mc_grid = np.linspace(10, 50, 200)
d_grid = np.linspace(100, 1000, 181)

t_model = t_chirp(freq[:, None], mc_grid[None, :])
res_t = (t_obs[:, None] - t_model) / sigma_t
loglike_t = -0.5 * np.sum(res_t**2, axis=0)

h_model = h_amp(freq[:, None, None], mc_grid[None, :, None], d_grid[None, None, :])
res_h = (h_obs[:, None, None] - h_model) / sigma_h[:, None, None]
loglike_h = -0.5 * np.sum(res_h**2, axis=0)

loglike = loglike_t[:, None] + loglike_h

prior_mc = np.ones_like(mc_grid) / (mc_grid[-1] - mc_grid[0])
prior_d = d_grid**2
prior_d /= np.trapz(prior_d, d_grid)

log_prior = np.log(prior_mc)[:, None] + np.log(prior_d)[None, :]

log_post = loglike + log_prior
log_post -= np.max(log_post)

posterior = np.exp(log_post)
norm = np.trapz(np.trapz(posterior, d_grid, axis=1), mc_grid)
posterior /= norm


---
## 6. Mapa posterior y contornos de credibilidad


In [ ]:
MC, DL = np.meshgrid(mc_grid, d_grid, indexing="ij")

def contour_levels(pdf, levels=(0.5, 0.9)):
    sorted_pdf = np.sort(pdf.ravel())[::-1]
    cdf = np.cumsum(sorted_pdf)
    cdf /= cdf[-1]
    return [sorted_pdf[np.searchsorted(cdf, level)] for level in levels]

levels = contour_levels(posterior, levels=(0.5, 0.9))

plt.figure(figsize=(6, 5))
plt.pcolormesh(mc_grid, d_grid, posterior.T, shading="auto")
plt.contour(mc_grid, d_grid, posterior.T, levels=sorted(levels), colors="white")
plt.plot(mc_true, d_true, "r*", markersize=12, label="Valor verdadero")
plt.xlabel(r"$\mathcal{M}~[M_\odot]$")
plt.ylabel(r"$d_L~[\mathrm{Mpc}]$")
plt.title("Posterior 2D (prior volumétrico)")
plt.legend()
plt.colorbar(label="p(\mathcal{M}, d_L | D)")
plt.tight_layout()


---
## 7. Distribuciones marginales e intervalos creíbles


In [ ]:
p_mc = np.trapz(posterior, d_grid, axis=1)
p_d = np.trapz(posterior, mc_grid, axis=0)

def credible_interval(x, pdf, level=0.9):
    dx = np.gradient(x)
    cdf = np.cumsum(pdf * dx)
    cdf /= cdf[-1]
    lo = np.interp((1 - level) / 2, cdf, x)
    hi = np.interp(1 - (1 - level) / 2, cdf, x)
    med = np.interp(0.5, cdf, x)
    return med, lo, hi

mc_med, mc_lo, mc_hi = credible_interval(mc_grid, p_mc, level=0.9)
d_med, d_lo, d_hi = credible_interval(d_grid, p_d, level=0.9)

print(f"Masa chirp (mediana, 90% CI): {mc_med:.2f} [{mc_lo:.2f}, {mc_hi:.2f}] M_sun")
print(f"Distancia (mediana, 90% CI): {d_med:.1f} [{d_lo:.1f}, {d_hi:.1f}] Mpc")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(mc_grid, p_mc)
axes[0].axvline(mc_med, color="k", ls="--")
axes[0].set_xlabel(r"$\mathcal{M}~[M_\odot]$")
axes[0].set_ylabel("p(\mathcal{M}|D)")
axes[0].set_title("Marginal en masa")

axes[1].plot(d_grid, p_d)
axes[1].axvline(d_med, color="k", ls="--")
axes[1].set_xlabel(r"$d_L~[\mathrm{Mpc}]$")
axes[1].set_ylabel("p(d_L|D)")
axes[1].set_title("Marginal en distancia")

plt.tight_layout()


---
## 8. Probabilidades de interés


In [ ]:
prob_mc_gt_30 = np.trapz(p_mc[mc_grid > 30], mc_grid[mc_grid > 30])
prob_d_lt_500 = np.trapz(p_d[d_grid < 500], d_grid[d_grid < 500])

print(f"P(M_c > 30 M_sun) = {prob_mc_gt_30:.3f}")
print(f"P(d_L < 500 Mpc) = {prob_d_lt_500:.3f}")


---
## 9. Sensibilidad al prior de distancia


In [ ]:
prior_d_uniform = np.ones_like(d_grid) / (d_grid[-1] - d_grid[0])
log_prior_uniform = np.log(prior_mc)[:, None] + np.log(prior_d_uniform)[None, :]

log_post_uniform = loglike + log_prior_uniform
log_post_uniform -= np.max(log_post_uniform)
posterior_uniform = np.exp(log_post_uniform)
posterior_uniform /= np.trapz(np.trapz(posterior_uniform, d_grid, axis=1), mc_grid)

p_d_uniform = np.trapz(posterior_uniform, mc_grid, axis=0)

d_med_u, d_lo_u, d_hi_u = credible_interval(d_grid, p_d_uniform, level=0.9)
print(f"Distancia (prior uniforme) mediana 90% CI: {d_med_u:.1f} [{d_lo_u:.1f}, {d_hi_u:.1f}] Mpc")

plt.figure(figsize=(6, 4))
plt.plot(d_grid, p_d, label="Prior volumétrico")
plt.plot(d_grid, p_d_uniform, label="Prior uniforme")
plt.xlabel(r"$d_L~[\mathrm{Mpc}]$")
plt.ylabel("p(d_L|D)")
plt.title("Efecto del prior en distancia")
plt.legend()
plt.tight_layout()


---
## 10. Conclusiones
- La inferencia bayesiana permite cuantificar incertidumbres reales en parámetros físicos.
- La masa chirp está fuertemente determinada por la evolución temporal de la frecuencia.
- La distancia es más sensible al prior, reflejando la degeneración con la amplitud.

Este flujo de trabajo es una versión simplificada pero realista de cómo LVK estima parámetros a partir de datos observacionales.
